# 01 · Download and prepare the data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imbishal7/oil-slick-segmentation/blob/main/notebooks/01_prepare_data.ipynb)

Downloads the Sentinel-1 oil spill dataset (Trujillo-Acatitla et al., 2024; Zenodo
[8346860](https://zenodo.org/records/8346860), [8253899](https://zenodo.org/records/8253899),
[13761290](https://zenodo.org/records/13761290)) and turns it into model-ready arrays on Google Drive.

**Runtime:** CPU is enough. It needs about 45 GB of free local disk and about 7 GB on Drive.

What `oilslick.prepare` does:
1. **Per archive:** downloads it to local disk and checks its MD5. It then streams every GeoTIFF out of
   it in memory, downsamples it from 2048 to 512 px and caches the result to
   `PROJECT_DIR/cache/<archive>.npz`, then deletes the archive. Full-resolution images never touch the
   disk. About 96 GB is downloaded in total.
2. **Assemble:** pairs images with masks and audits them. It splits train/val 80/20, stratified by
   category, so that patches from the same Sentinel-1 product or with overlapping footprints never
   straddle the split. It then computes the normalization statistics on train only.

**Resumable:** after a disconnect, run all cells again. Cached archives are skipped and
interrupted downloads resume.

In [ ]:
REPO_URL = "https://github.com/imbishal7/oil-slick-segmentation.git"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/oil-slick-segmentation"  # everything persistent lives here

In [ ]:
# Mount Drive, fetch the latest code and install it.
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_DIR = "/content/oil-slick-segmentation"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --format="code at %h: %s"
!pip install -q -e {REPO_DIR}
sys.path.insert(0, f"{REPO_DIR}/src")  # importable without restarting the runtime
os.makedirs(PROJECT_DIR, exist_ok=True)

In [ ]:
!df -h /content | tail -1  # local disk: needs ~45 GB free for the largest archive

## Download, downsample, split

In [ ]:
from oilslick import prepare

prepare.main([
    "--scratch-dir", "/content/scratch",          # local disk, temporary
    "--cache-dir", f"{PROJECT_DIR}/cache",        # per-archive cache on Drive (~3.5 GB)
    "--out-dir", f"{PROJECT_DIR}/data",           # final arrays for notebook 02 (~3.5 GB)
])

## Audit

In [ ]:
import json
import pandas as pd

audit = json.load(open(f"{PROJECT_DIR}/data/audit.json"))
display(pd.DataFrame(audit["counts"]).T.assign(total=lambda d: d.sum(axis=1)))
{k: v for k, v in audit.items() if k != "counts"}

## Sanity check: a few training patches

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data_dir = f"{PROJECT_DIR}/data"
manifest = pd.read_csv(f"{data_dir}/manifest.csv")
images = np.load(f"{data_dir}/train_images.npy", mmap_mode="r")
masks = np.load(f"{data_dir}/train_masks.npy", mmap_mode="r")

train = manifest[manifest["split"] == "train"]
examples = train.groupby("category").sample(2, random_state=0)
fig, axes = plt.subplots(3, len(examples), figsize=(3 * len(examples), 9))
for col, (_, row) in enumerate(examples.iterrows()):
    vv, vh = images[row["index"]].astype(np.float32)
    panels = {f"{row['category']} · VV": vv, "VH": vh, "mask": masks[row["index"]]}
    for ax, (title, panel) in zip(axes[:, col], panels.items()):
        ax.imshow(panel, cmap="gray")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
plt.tight_layout()

## Optional: free Drive space

The per-archive cache is only needed to re-assemble (for example, with a different split seed) without
downloading again. Delete it once you are happy with the data.

In [ ]:
# !rm -rf "{PROJECT_DIR}/cache"